In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import glob
from stable_baselines3.common.results_plotter import load_results

log_dir = r"C:\Users\shane\Documents\rlRobot\logs\\"
monitor_files = glob.glob(os.path.join(log_dir, "*.monitor.csv"))

runs = []
for f in monitor_files:
    df = pd.read_csv(f, skiprows=1, on_bad_lines='skip')
    df["file"] = os.path.basename(f)
    runs.append(df)

data = pd.concat(runs, ignore_index=True)
data = data.sort_values("t").reset_index(drop=True)
data["episode"] = data.index

def smooth(y, window=50): return pd.Series(y).rolling(window, min_periods=1).mean()

# reward over episodes
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Louis analysis", fontsize=16, fontweight="bold")

ax = axes[0, 0]
ax.plot(data["episode"], data["r"], alpha=0.2, color="orange", label="raw")
ax.plot(data["episode"], smooth(data["r"]), color="darkorange", linewidth=2, label="smoothed")
ax.set_title("Episode Reward")
ax.set_xlabel("Episode")
ax.set_ylabel("Reward")
ax.legend()
ax.grid(alpha=0.3)

# episode length over episodes
ax = axes[0, 1]
ax.plot(data["episode"], data["l"], alpha=0.2, color="steelblue", label="raw")
ax.plot(data["episode"], smooth(data["l"]), color="steelblue", linewidth=2, label="smoothed")
ax.axhline(5000, color="green", linestyle="--", alpha=0.5, label="max (5000)")
ax.set_title("Episode Length (Survival)")
ax.set_xlabel("Episode")
ax.set_ylabel("Steps")
ax.legend()
ax.grid(alpha=0.3)

# reward distribution
ax = axes[1, 0]
ax.hist(data["r"], bins=50, color="orange", edgecolor="darkorange", alpha=0.8)
ax.set_title("Reward Distribution")
ax.set_xlabel("Reward")
ax.set_ylabel("Count")
ax.grid(alpha=0.3)

# reward vs episode length scatter
ax = axes[1, 1]
sc = ax.scatter(data["l"], data["r"], alpha=0.3, c=data["episode"], cmap="plasma", s=5)
plt.colorbar(sc, ax=ax, label="episode")
ax.set_title("Reward vs Survival Time")
ax.set_xlabel("Episode Length")
ax.set_ylabel("Reward")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(r"C:\Users\shane\Documents\rlRobot\logs\training_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

# summary stats
print("=" * 40)
print("Training Summary From All Sessions")
print("=" * 40)
print(f"Total episodes:       {len(data)}")
print(f"Total timesteps:      {int(data['t'].max()):,}")
print(f"Best reward:          {data['r'].max():.1f}")
print(f"Mean reward:          {data['r'].mean():.1f}")
print(f"Mean episode length:  {data['l'].mean():.0f} steps")
print(f"Max episode length:   {data['l'].max()} steps")
print(f"Episodes at max len:  {(data['l'] == 5000).sum()}")
print("=" * 40)

Training Summary From All Sessions
Total episodes:       529727
Total timesteps:      42,506
Best reward:          35831.7
Mean reward:          1388.8
Mean episode length:  687 steps
Max episode length:   5000 steps
Episodes at max len:  10804
